# Phase 2: Feature Engineering and Resampling

This notebook builds the Final Feature Matrix from the Gold Dataset (8,268 seniors) using hrp_processed.db. We'll:

1. **Resample** vitals into 15-minute buckets (mean for HR/BP/Temp/Sat, sum for Steps)
2. **Pivot** from long to wide format
3. **Impute** missing values with forward fill (1 hour limit)
4. **Engineer features**: hr_volatility, bp_trend, pulse_pressure
5. **Fuse static data**: 11 clinical domain flags
6. **Create alert context**: recent_event_burden (Severity 1/2 alerts in past 48h)
7. **Label targets**: label_3 = 1 if within 24h before Severity 3 alert
8. **Save output**: data/processed/multimodal_features.parquet

Processing senior-by-senior to manage memory for the large final matrix.

## Section 1: Import Required Libraries

In [6]:
import time
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import pyarrow.parquet as pq
from scipy import stats
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

## Section 2: Load Gold Dataset and Database Connection

In [7]:
db_path = '../data/processed/hrp_processed.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Connected to: {db_path}")

Connected to: ../data/processed/hrp_processed.db


In [8]:
# Load Gold Dataset from previous notebook analysis
# Gold Dataset: 8,268 seniors meeting >30% density OR >80% local density before Severity 3 alerts

import pickle
import json

gold_dataset_pickle_path = '../data/processed/gold_seniors.pkl'
gold_dataset_csv_path = '../data/processed/gold_seniors.csv'
metadata_path = '../data/processed/gold_dataset_metadata.json'

# Load the gold seniors set from pickle
with open(gold_dataset_pickle_path, 'rb') as f:
    gold_seniors = pickle.load(f)

print(f"✓ Loaded Gold Dataset from: {gold_dataset_pickle_path}")
print(f"  Total seniors in Gold Dataset: {len(gold_seniors):,}")

# Load metadata
with open(metadata_path, 'r') as f:
    gold_metadata = json.load(f)

print(f"\n✓ Gold Dataset Metadata:")
print(f"  - Total Seniors: {gold_metadata['total_seniors']:,}")
print(f"  - Retention Rate: {gold_metadata['retention_rate_percent']:.1f}%")
print(f"  - Severity 3 Alerts Retained: {gold_metadata['severity_3_alerts']:,} ({gold_metadata['severity_3_retention_percent']:.1f}%)")
print(f"  - Criteria:")
print(f"    * Criterion 1 ({gold_metadata['criterion_1_count']:,}): {gold_metadata['criteria_description']['criterion_1']}")
print(f"    * Criterion 2 ({gold_metadata['criterion_2_count']:,}): {gold_metadata['criteria_description']['criterion_2']}")

# Convert to list for processing
gold_seniors_list = list(gold_seniors)
print(f"\n✓ Ready to process {len(gold_seniors_list):,} seniors")

✓ Loaded Gold Dataset from: ../data/processed/gold_seniors.pkl
  Total seniors in Gold Dataset: 8,268

✓ Gold Dataset Metadata:
  - Total Seniors: 8,268
  - Retention Rate: 64.6%
  - Severity 3 Alerts Retained: 47 (77.0%)
  - Criteria:
    * Criterion 1 (8,267): Seniors with >30% overall data density
    * Criterion 2 (26): Seniors with >80% local density in 6 hours before Severity 3 alert

✓ Ready to process 8,268 seniors


## CRITICAL: Create Database Index for Performance

Before processing 8,268 seniors with 70M+ measurements, we need an index on senior_id to avoid full table scans.

In [9]:
# Create index on measurements.senior_id to speed up queries from ~minutes to milliseconds
# This is essential for processing 8,268 seniors efficiently

print("Checking for existing index on measurements.senior_id...")
index_check = cursor.execute("SELECT name FROM sqlite_master WHERE type='index' AND tbl_name='measurements'").fetchall()
print(f"Existing indexes on measurements table: {[idx[0] for idx in index_check]}")

# Create index if it doesn't exist
print("\nCreating index on senior_id (this may take 1-2 minutes for 70M rows)...")
try:
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_measurements_senior_id ON measurements(senior_id)")
    conn.commit()
    print("✓ Index created successfully!")
except Exception as e:
    print(f"Index creation: {e}")

# Verify index exists
index_verify = cursor.execute("SELECT name FROM sqlite_master WHERE type='index' AND tbl_name='measurements'").fetchall()
print(f"\nCurrent indexes: {[idx[0] for idx in index_verify]}")

Checking for existing index on measurements.senior_id...
Existing indexes on measurements table: ['idx_measurements_senior', 'idx_measurements_date', 'idx_measurements_senior_date']

Creating index on senior_id (this may take 1-2 minutes for 70M rows)...
✓ Index created successfully!

Current indexes: ['idx_measurements_senior', 'idx_measurements_date', 'idx_measurements_senior_date', 'idx_measurements_senior_id']


## Section 3: Resample Vitals into 15-Minute Buckets

In [ ]:
def resample_senior_vitals(senior_id, conn, verbose=False):
    """
    Resample a single senior's measurements into 15-minute buckets.
    
    Returns DataFrame with 15-min buckets as rows, vitals as columns.
    - Mean aggregation for: HR, SBP, DBP, Temperature, Saturation
    - Sum aggregation for: Steps
    
    Note: BloodPressure type stores values in sbp and dbp columns (not value column)
    """
    
    if verbose:
        # Quick count first to diagnose performance (uses index on senior_id)
        count_query = "SELECT COUNT(*) as c FROM measurements WHERE senior_id = ?"
        count_result = pd.read_sql_query(count_query, conn, params=(str(senior_id),))
        row_count = count_result['c'][0]
        print(f"      → Found {row_count:,} measurement rows for senior {senior_id}")
        if row_count > 50000:
            print(f"      ⚠ Large dataset detected - this may take a while...")
    
    # Load all measurements for this senior, including sbp and dbp for BloodPressure
    query = """
    SELECT date, type, value, sbp, dbp FROM measurements 
    WHERE senior_id = ?
    ORDER BY date
    """
    
    df = pd.read_sql_query(query, conn, params=(str(senior_id),))
    
    if len(df) == 0:
        return None
    
    df['date'] = pd.to_datetime(df['date'])
    
    # Create mapping for aggregation functions
    agg_funcs = {
        'Heartrate': 'mean',
        'Temperature': 'mean',
        'Saturation': 'mean',
        'Steps': 'sum'
    }
    
    # Rename type to measurement_type for clarity
    df.rename(columns={'type': 'measurement_type'}, inplace=True)
    
    # Set the date as index for resampling
    df.set_index('date', inplace=True)
    
    # Group by measurement type and resample each to 15-minute buckets
    resampled_dfs = []
    
    for mtype in df['measurement_type'].unique():
        df_type = df[df['measurement_type'] == mtype].copy()
        
        # Handle BloodPressure specially - values are in sbp and dbp columns
        if mtype == 'BloodPressure':
            # Resample sbp and dbp separately to 15 minutes with mean
            resampled_sbp = df_type['sbp'].resample('15min').mean()
            resampled_dbp = df_type['dbp'].resample('15min').mean()
            resampled_dfs.append(resampled_sbp.to_frame(name='sbp'))
            resampled_dfs.append(resampled_dbp.to_frame(name='dbp'))
        else:
            # For other types, use the value column
            agg_func = agg_funcs.get(mtype, 'mean')
            resampled = df_type['value'].resample('15min').agg(agg_func)
            resampled_dfs.append(resampled.to_frame(name=mtype.lower()))
    
    # Combine all resampled measurements
    if len(resampled_dfs) > 0:
        result = pd.concat(resampled_dfs, axis=1)
        result.reset_index(inplace=True)
        result['senior_id'] = senior_id
        return result
    else:
        return None

print("Resample function defined")

Resample function defined


## Section 4: Pivot Data from Long to Wide Format

In [11]:
def pivot_to_wide_format(df_resampled):
    """
    Transform resampled data from long to wide format.
    Each time bucket becomes a row with columns for all vital signs.
    """
    
    if df_resampled is None or len(df_resampled) == 0:
        return None
    
    # The resampled dataframe should already be in wide format
    # (one row per bucket with vital columns)
    # Just ensure proper structure
    
    df_wide = df_resampled.copy()
    df_wide.rename(columns={'date': 'timestamp'}, inplace=True)
    
    # Clean column names - lowercase and underscored
    df_wide.columns = [col.lower().replace(' ', '_') for col in df_wide.columns]
    
    return df_wide

print("Pivot function defined")

Pivot function defined


## Section 5: Apply Forward Fill Imputation

In [12]:
def apply_forward_fill_imputation(df_wide, limit_buckets=4):
    """
    Apply forward fill imputation with a limit of 4 buckets (1 hour at 15-min intervals).
    This fills small gaps while preserving data integrity.
    
    limit_buckets=4 means max 1 hour of forward fill (4 * 15 min = 60 min)
    """
    
    if df_wide is None or len(df_wide) == 0:
        return df_wide
    
    df_filled = df_wide.copy()
    
    # List of vital columns to fill (exclude timestamp and senior_id)
    vital_cols = [col for col in df_filled.columns 
                  if col not in ['timestamp', 'senior_id']]
    
    # Apply forward fill with limit for each vital
    for col in vital_cols:
        df_filled[col] = df_filled[col].fillna(method='ffill', limit=limit_buckets)
    
    return df_filled

print("Forward fill imputation function defined")

Forward fill imputation function defined


## Section 6: Engineer Signal Features (Volatility, Trend, Pulse Pressure)

In [13]:
def engineer_signal_features(df_filled):
    """
    Add engineered signal features:
    - hr_volatility: 4-hour rolling standard deviation of HR (16 buckets at 15-min)
    - bp_trend: Slope of SBP over last 3 hours (12 buckets at 15-min)
    - pulse_pressure: SBP - DBP
    """
    
    if df_filled is None or len(df_filled) == 0:
        return df_filled
    
    df_features = df_filled.copy()
    
    # HR volatility: 4-hour rolling std (16 buckets * 15 min = 240 min = 4 hours)
    if 'heartrate' in df_features.columns:
        df_features['hr_volatility'] = df_features['heartrate'].rolling(
            window=16, min_periods=1
        ).std()
    
    # Blood pressure features based on SBP/DBP columns
    def calculate_slope(series):
        if len(series) < 2:
            return np.nan
        x = np.arange(len(series))
        mask = ~np.isnan(series)
        if mask.sum() < 2:
            return np.nan
        slope, _ = np.polyfit(x[mask], series[mask], 1)
        return slope
    
    # BP trend uses SBP
    if 'sbp' in df_features.columns:
        df_features['bp_trend'] = df_features['sbp'].rolling(
            window=12, min_periods=2
        ).apply(calculate_slope, raw=False)
    
    # Pulse pressure = SBP - DBP when both available
    if 'sbp' in df_features.columns and 'dbp' in df_features.columns:
        df_features['pulse_pressure'] = df_features['sbp'] - df_features['dbp']
    
    return df_features

print("Signal feature engineering function defined")

Signal feature engineering function defined


## Section 7: Fuse Static Clinical Domain Flags

In [ ]:
def load_clinical_domain_flags(senior_id, conn):
    """
    Load the 11 clinical domain flags from the senior_risk_profiles table.
    These are static features that apply to all time buckets for a senior.
    """
    
    query = """
    SELECT * FROM senior_risk_profiles WHERE senior_id = ?
    """
    
    df_flags = pd.read_sql_query(query, conn, params=(str(senior_id),))
    
    if len(df_flags) == 0:
        return None
    
    # Extract flag columns (assuming naming convention like is_*, has_*, flag_*)
    # Keep senior_id and any boolean/flag columns
    flag_cols = [col for col in df_flags.columns 
                 if col != 'senior_id' and df_flags[col].dtype in ['int64', 'float64', 'bool']]
    
    return df_flags[['senior_id'] + flag_cols].iloc[0]

def fuse_clinical_flags(df_features, clinical_flags):
    """
    Join clinical domain flags to each time bucket.
    Flags are static (same for all buckets of a senior).
    """
    
    if df_features is None or clinical_flags is None:
        return df_features
    
    df_fused = df_features.copy()
    
    # Add each flag column to every row
    for col in clinical_flags.index:
        if col != 'senior_id':
            df_fused[col] = clinical_flags[col]
    
    return df_fused

print("Clinical flags functions defined")

Clinical flags functions defined


## Section 8: Create Alert Context Feature (Recent Event Burden)

In [ ]:
def add_alert_context_feature(df_features, senior_id, conn):
    """
    Create recent_event_burden feature:
    For each time bucket, count the number of Severity 1 or 2 alerts
    that occurred in the 48 hours BEFORE that bucket.
    """
    
    if df_features is None or len(df_features) == 0:
        return df_features
    
    df_context = df_features.copy()
    
    # Load all Severity 1 & 2 alerts for this senior
    query = """
    SELECT alert_date FROM alerts 
    WHERE senior_id = ? AND severity IN (1, 2)
    ORDER BY alert_date
    """
    
    alerts_df = pd.read_sql_query(query, conn, params=(str(senior_id),))
    
    if len(alerts_df) == 0:
        # No alerts, all buckets have 0 burden
        df_context['recent_event_burden'] = 0
        return df_context
    
    alerts_df['alert_date'] = pd.to_datetime(alerts_df['alert_date'])
    
    # For each time bucket, count alerts in the past 48 hours
    def count_recent_alerts(timestamp, alert_dates):
        if pd.isna(timestamp):
            return 0
        window_start = timestamp - timedelta(hours=48)
        count = ((alert_dates >= window_start) & (alert_dates < timestamp)).sum()
        return count
    
    df_context['recent_event_burden'] = df_context['timestamp'].apply(
        lambda ts: count_recent_alerts(ts, alerts_df['alert_date'])
    )
    
    return df_context

print("Alert context feature function defined")

Alert context feature function defined


## Section 9: Label Target Variable (Severity 3 Alert Window)

In [ ]:
def create_target_label(df_features, senior_id, conn):
    """
    Create binary target label_3:
    - label_3 = 1 if the time bucket occurs within 24 hours BEFORE a Severity 3 alert
    - label_3 = 0 otherwise
    """
    
    if df_features is None or len(df_features) == 0:
        return df_features
    
    df_labeled = df_features.copy()
    df_labeled['label_3'] = 0  # Default to 0
    
    # Load all Severity 3 alerts for this senior
    query = """
    SELECT alert_date FROM alerts 
    WHERE senior_id = ? AND severity = 3
    """
    
    alerts_df = pd.read_sql_query(query, conn, params=(str(senior_id),))
    
    if len(alerts_df) == 0:
        # No Severity 3 alerts for this senior
        return df_labeled
    
    alerts_df['alert_date'] = pd.to_datetime(alerts_df['alert_date'])
    alert_dates = alerts_df['alert_date'].values
    
    # For each Severity 3 alert, mark all buckets in the 24-hour window before it
    for alert_ts in alert_dates:
        window_start = pd.Timestamp(alert_ts) - timedelta(hours=24)
        window_end = pd.Timestamp(alert_ts)
        
        # Mark buckets in this window
        mask = (df_labeled['timestamp'] >= window_start) & \
               (df_labeled['timestamp'] < window_end)
        df_labeled.loc[mask, 'label_3'] = 1
    
    return df_labeled

print("Target labeling function defined")

Target labeling function defined


## Section 10: Process Senior-by-Senior and Save to Parquet

In [17]:
# Gold Dataset seniors have already been loaded in Section 2
# They were saved from notebook 03 analysis

print("\n" + "=" * 100)
print("PROCESSING GOLD DATASET SENIORS")
print("=" * 100)
print(f"Total seniors to process: {len(gold_seniors_list):,}")
print(f"Sample seniors: {gold_seniors_list[:5]}")
print("=" * 100)


PROCESSING GOLD DATASET SENIORS
Total seniors to process: 8,268
Sample seniors: [np.int64(32776), np.int64(32788), np.int64(32821), np.int64(32832), np.int64(32834)]


In [ ]:
## Quick diagnostics: inspect one senior's measurements before full run
sample_senior = gold_seniors_list[0] if len(gold_seniors_list) > 0 else None
if sample_senior is not None:
    print(f"Sampling senior: {sample_senior}")
    diag_query = """
    SELECT senior_id, type, value, sbp, dbp, date
    FROM measurements
    WHERE senior_id = ?
    ORDER BY date
    LIMIT 5
    """
    diag_df = pd.read_sql_query(diag_query, conn, params=(str(sample_senior),))
    print(f"Rows fetched: {len(diag_df)}")
    print(diag_df.head())
else:
    print("Gold seniors list is empty; cannot sample.")


Sampling senior: 32776
Rows fetched: 5
   senior_id   type  value   sbp   dbp                 date
0      32776  Steps   28.0  None  None  2025-11-01 08:11:36
1      32776  Steps   38.0  None  None  2025-11-01 08:17:36
2      32776  Steps   77.0  None  None  2025-11-01 08:20:36
3      32776  Steps  174.0  None  None  2025-11-01 08:23:36
4      32776  Steps  187.0  None  None  2025-11-01 08:26:36


In [ ]:
# Coverage diagnostics: do we have measurements for any gold senior?
print("\n=== COVERAGE DIAGNOSTICS ===")
total_measurements = pd.read_sql_query("SELECT COUNT(*) AS c FROM measurements", conn)['c'][0]
distinct_measurement_seniors = pd.read_sql_query("SELECT COUNT(DISTINCT senior_id) AS c FROM measurements", conn)['c'][0]
print(f"Total rows in measurements: {total_measurements:,}")
print(f"Distinct seniors in measurements: {distinct_measurement_seniors:,}")
print(f"Gold seniors count: {len(gold_seniors_list):,}")

# Fetch measurement senior ids and compute overlap in Python (avoids huge SQL IN)
measurement_seniors = set(pd.read_sql_query("SELECT DISTINCT senior_id FROM measurements", conn)['senior_id'])
gold_seniors_set = set(gold_seniors_list)
overlap_seniors = list(measurement_seniors & gold_seniors_set)
print(f"Overlap seniors between measurements and gold list: {len(overlap_seniors):,}")

# If overlap exists, sample one and preview rows
if len(overlap_seniors) > 0:
    sample_overlap = overlap_seniors[0]
    print(f"Sampling overlapping senior: {sample_overlap}")
    preview_df = pd.read_sql_query(
        """
        SELECT senior_id, type, value, sbp, dbp, date
        FROM measurements
        WHERE senior_id = ?
        ORDER BY date
        LIMIT 5
        """,
        conn,
        params=(str(sample_overlap),)
    )
    print(f"Rows fetched: {len(preview_df)}")
    print(preview_df.head())
else:
    print("No overlap between gold seniors and measurements table. Confirm database path and gold list source.")



=== COVERAGE DIAGNOSTICS ===
Total rows in measurements: 70,428,508
Distinct seniors in measurements: 12,823
Gold seniors count: 8,268
Overlap seniors between measurements and gold list: 8,268
Sampling overlapping senior: 32776
Rows fetched: 5
   senior_id   type  value   sbp   dbp                 date
0      32776  Steps   28.0  None  None  2025-11-01 08:11:36
1      32776  Steps   38.0  None  None  2025-11-01 08:17:36
2      32776  Steps   77.0  None  None  2025-11-01 08:20:36
3      32776  Steps  174.0  None  None  2025-11-01 08:23:36
4      32776  Steps  187.0  None  None  2025-11-01 08:26:36


In [20]:
def process_senior_complete_pipeline(senior_id, conn, verbose=False):
    """
    Complete pipeline for a single senior:
    1. Resample to 15-min buckets
    2. Pivot to wide format
    3. Apply forward fill imputation
    4. Engineer signal features
    5. Fuse clinical flags
    6. Add alert context
    7. Create target label
    """
    
    try:
        if verbose:
            print(f"    [Step 1/7] Resampling vitals...")
        # Step 1: Resample
        df_resampled = resample_senior_vitals(senior_id, conn, verbose=verbose)
        if df_resampled is None or len(df_resampled) == 0:
            if verbose:
                print(f"    [Step 1/7] No measurements found - skipping")
            return None
        if verbose:
            print(f"    [Step 1/7] ✓ Resampled to {len(df_resampled)} buckets")
        
        if verbose:
            print(f"    [Step 2/7] Pivoting to wide format...")
        # Step 2: Pivot (already in wide format from resampling)
        df_wide = pivot_to_wide_format(df_resampled)
        if df_wide is None:
            return None
        if verbose:
            print(f"    [Step 2/7] ✓ Complete")
        
        if verbose:
            print(f"    [Step 3/7] Forward fill imputation...")
        # Step 3: Forward fill imputation
        df_filled = apply_forward_fill_imputation(df_wide)
        if verbose:
            print(f"    [Step 3/7] ✓ Complete")
        
        if verbose:
            print(f"    [Step 4/7] Engineering signal features...")
        # Step 4: Engineer signal features
        df_signals = engineer_signal_features(df_filled)
        if verbose:
            print(f"    [Step 4/7] ✓ Complete")
        
        if verbose:
            print(f"    [Step 5/7] Loading clinical flags...")
        # Step 5: Fuse clinical flags
        clinical_flags = load_clinical_domain_flags(senior_id, conn)
        if clinical_flags is not None:
            df_signals = fuse_clinical_flags(df_signals, clinical_flags)
            if verbose:
                print(f"    [Step 5/7] ✓ Flags loaded and fused")
        else:
            if verbose:
                print(f"    [Step 5/7] ⚠ No clinical flags found")
        
        if verbose:
            print(f"    [Step 6/7] Adding alert context (48h burden)...")
        # Step 6: Add alert context
        df_context = add_alert_context_feature(df_signals, senior_id, conn)
        if verbose:
            print(f"    [Step 6/7] ✓ Complete")
        
        if verbose:
            print(f"    [Step 7/7] Creating target labels...")
        # Step 7: Create target label
        df_final = create_target_label(df_context, senior_id, conn)
        if verbose:
            print(f"    [Step 7/7] ✓ Complete")
        
        return df_final
    
    except Exception as e:
        print(f"Error processing senior {senior_id}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

print("Complete pipeline function defined")

Complete pipeline function defined


In [ ]:
print("=" * 100)
print("BUILDING FINAL FEATURE MATRIX")
print("=" * 100)

# Process seniors in batches to manage memory
batch_size = 100
all_features_list = []
successful_seniors = 0
failed_seniors = 0

output_path = '../data/processed/multimodal_features.parquet'

start_time = time.time()

# Enable verbose mode for first 3 seniors to diagnose issues
for idx, senior_id in enumerate(gold_seniors_list):
    senior_start = time.time()
    verbose = (idx < 3)  # Verbose output for first 3 seniors only
    
    # More frequent progress updates, especially at the start
    if idx < 10 or (idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f"Processing senior {idx + 1}/{len(gold_seniors_list)} (ID: {senior_id}) | Elapsed: {elapsed/60:.1f}min | Success: {successful_seniors} | Failed: {failed_seniors}")
    
    # Process this senior
    df_senior = process_senior_complete_pipeline(senior_id, conn, verbose=verbose)
    
    if df_senior is not None and len(df_senior) > 0:
        all_features_list.append(df_senior)
        successful_seniors += 1
        senior_elapsed = time.time() - senior_start
        if idx < 5:  # Show timing for first few seniors
            print(f"  ✓ Senior {senior_id} processed in {senior_elapsed:.2f}s - {len(df_senior)} time buckets")
    else:
        failed_seniors += 1
        if idx < 10:  # Show failures for first 10
            print(f"  ✗ Senior {senior_id} returned no data")

print(f"\n✓ Successfully processed: {successful_seniors:,} seniors")
print(f"✗ Failed/skipped: {failed_seniors:,} seniors")

# Combine all seniors into one large dataframe
if len(all_features_list) > 0:
    df_final_matrix = pd.concat(all_features_list, ignore_index=True)
    print(f"\nFinal Feature Matrix Shape: {df_final_matrix.shape}")
    print(f"Total Time Buckets: {len(df_final_matrix):,}")
    print(f"Unique Seniors: {df_final_matrix['senior_id'].nunique():,}")
    
    # Display column information
    print(f"\nFeature Columns ({len(df_final_matrix.columns)}):")
    for col in df_final_matrix.columns:
        print(f"  - {col}")
    
    # Display target distribution
    print(f"\nTarget Variable Distribution (label_3):")
    print(df_final_matrix['label_3'].value_counts())
    
    print(f"\nData types:")
    print(df_final_matrix.dtypes)
else:
    print("No seniors were successfully processed!")

BUILDING FINAL FEATURE MATRIX
Processing senior 1/8268 (ID: 32776) | Elapsed: 0.0min | Success: 0 | Failed: 0
    [Step 1/7] Resampling vitals...
      → Found 1,025 measurement rows for senior 32776


In [ ]:
# Save to Parquet
if len(all_features_list) > 0:
    print("\n" + "=" * 100)
    print("SAVING TO PARQUET")
    print("=" * 100)
    
    # Convert timestamp to string for better parquet compatibility
    df_final_matrix['timestamp'] = df_final_matrix['timestamp'].astype(str)
    
    # Save to parquet format (compressed)
    df_final_matrix.to_parquet(output_path, compression='snappy', index=False)
    
    print(f"✓ Feature matrix saved to: {output_path}")
    print(f"✓ File size: {np.round(pd.io.common.get_filepath_or_buffer(output_path)[0].__sizeof__() / (1024**3), 2)} GB")
    
    # Verify the file
    df_verify = pd.read_parquet(output_path)
    print(f"\n✓ Verification:")
    print(f"  - Rows: {len(df_verify):,}")
    print(f"  - Columns: {len(df_verify.columns)}")
    print(f"  - Memory usage: {df_verify.memory_usage(deep=True).sum() / (1024**3):.2f} GB")
    
    # Summary statistics
    print(f"\n" + "=" * 100)
    print("FINAL FEATURE MATRIX SUMMARY")
    print("=" * 100)
    print(f"Total Rows (Time Buckets): {len(df_verify):,}")
    print(f"Unique Seniors: {df_verify['senior_id'].nunique():,}")
    print(f"Date Range: {df_verify['timestamp'].min()} to {df_verify['timestamp'].max()}")
    print(f"\nTarget Variable (label_3) Distribution:")
    label_counts = df_verify['label_3'].value_counts()
    for label, count in label_counts.items():
        print(f"  label_3 = {label}: {count:,} ({100*count/len(df_verify):.2f}%)")
else:
    print("Cannot save - no features were generated.")

## Section 11: Feature Matrix Exploration and Statistics

In [ ]:
if len(all_features_list) > 0:
    # Load the saved parquet file for exploration
    df_matrix = pd.read_parquet(output_path)
    
    print("=" * 100)
    print("FEATURE MATRIX DETAILED EXPLORATION")
    print("=" * 100)
    
    # Data types summary
    print("\nData Type Summary:")
    print(df_matrix.dtypes.value_counts())
    
    # Missing values
    print("\nMissing Values Summary:")
    missing = df_matrix.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0].sort_values(ascending=False))
    else:
        print("No missing values!")
    
    # Feature statistics
    print("\nVital Signs Statistics:")
    vital_cols = [col for col in df_matrix.columns 
                  if col in ['heartrate', 'temperature', 'saturation', 'steps', 'sbp', 'dbp']]
    if len(vital_cols) > 0:
        print(df_matrix[vital_cols].describe())
    
    # Engineered features statistics
    print("\nEngineered Features Statistics:")
    engineered_cols = [col for col in df_matrix.columns 
                       if col in ['hr_volatility', 'bp_trend', 'pulse_pressure', 'recent_event_burden']]
    if len(engineered_cols) > 0:
        print(df_matrix[engineered_cols].describe())
    
    # Target label statistics
    print("\nTarget Label Statistics (label_3):")
    print(f"  Total samples: {len(df_matrix):,}")
    print(f"  Positive cases (label_3=1): {(df_matrix['label_3'] == 1).sum():,} ({100*(df_matrix['label_3'] == 1).sum()/len(df_matrix):.2f}%)")
    print(f"  Negative cases (label_3=0): {(df_matrix['label_3'] == 0).sum():,} ({100*(df_matrix['label_3'] == 0).sum()/len(df_matrix):.2f}%)")
    print(f"  Class balance ratio: 1:{(df_matrix['label_3'] == 0).sum() / max(1, (df_matrix['label_3'] == 1).sum()):.2f}")
    
    print("\n" + "=" * 100)
    print("FEATURE ENGINEERING COMPLETE")
    print("=" * 100)
    print(f"✓ Output file: {output_path}")
    print(f"✓ Total samples: {len(df_matrix):,}")
    print(f"✓ Total features: {len(df_matrix.columns)}")
    print(f"✓ Seniors processed: {df_matrix['senior_id'].nunique():,}")
    
    conn.close()
    print("\n✓ Database connection closed")